**Contributed workflow — live GeoLab validation is still required.**

Read [README.md](README.md) first. Use a dedicated database and a small date subset. These notebooks can modify database collections and write S3 objects. No CSS data or credentials are distributed here.

# Load ANF data
This workflow is driven by a set of CSS3.0 relational database tables created with great effort by analysts at the Earthscope Array Network Facility during the operation of the Earthscope Transportable Array (TA).   The permanent archive for these data can be found [here](https://doi.org/10.17611/DP/EB.1).  From that top-level you browse to "anf" directory.  That large web page has arrival database tables by month of TA operation.   To build a master arrival database for building a usarray data set, however, it was far more convenient to download the merge of all these bundled into a single tar file.   That tar file is called "TA_Event_DB.tar".  Note again it is found in "anf" directory one in the doi link above.   A more volatile link is [this one](https://gage-data.earthscope.org/archive/seismology/products/event_bulletins/anf/).  I did some external work on my desktop to reduce the size of that database.   That reduction is just limiting the time range to when the TA was operating in the lower 48.  Deployment pages say the last official TA station in the lower 48 ended operation in September 2015.   The first monthly is June 2004 so I kept all arrival data from June 2004 through the end of September 2015.   That operation is a simple set of shell commands easily recreated.  The downloaded tar file is an image of that entire anf directory tree sans the tar file itself.   I took apart the tar file into a scratch work area and then ran this in that directory:
```
tables=(arrival assoc event netmag origin stamag)
for t in "${tables[@]}"; do
    echo Working on relation=$t
    cat 200*/*.$t > usarray48.$t
    cat 201[0-4]_??/*.$t >> usarray48.$t
    cat 2015_0?/*.$t >> usarray48.$t
done
```
Noting I explicitly dropped wfmeas as it wasn't consistently used and we don't need it for our study.   I also dropped origerr as we don't need that here either and it would just be baggage.   I downloaded that data to the css3.0 directory found in this work area.

There is one additional gotcha.   I had to obtain the final "snetsta" table from Frank Vernon who was the PI for the Earthscope ANF.   Earthscope, for some unknown reason, has not archived that table.  It is a bit of a detail but they are needed in this workflow to sort out the mismatch in concept between how css3.0 defines a station name and seed defines a station name.   If anyone is reading this who needs to reproduce this workflow you will need that table.   If this work ever results in publicationw we will need to supply that file.  The data handled by that shell script above is not trivial as it totals around 5 G.  

The data are tabulated in a set of text files that are used for data storage in Antelope's relational database system that ANF used for their operation.   MsPASS has a special python class to handle that type of data.   Hence, we first have to instantiate an instance of the what we call the DatascopeDatabase class.   The %env magic is needed to tell the DatascopeDatabase constructor how to load its dictionary of what attributes are in what table.   The need for that is an oddity of the setup here on GeoLab.   Oh, actually that file had to be copied from the master.  i.e. DatascopeDatabase.pf is in the mspass distribution data/pf directory.   Here I copied that to a local ".pf" directory.

In [ ]:
from mspasspy.preprocessing.css30.datascope import DatascopeDatabase
anfdb = DatascopeDatabase("css30/usarray48",pffile="./pf/DatascopeDatabase.pf")

This does all the dirty wrk to build a single DataFrame of what we need except the orid==prefor subsetting.  That is done later in this workflow.

In [ ]:
df = anfdb.CSS30Catalog2df()

In [ ]:
print(len(df))
print(df.keys())

Unassociated arrival entries turn out to leave NaN values in the evid and orid fields.  That causes pandas to change the type from int to float.  I think the reason is there is no natural undefined value in a dataframe for a int.   The next line fixes that by deleting all tuples for which evid or orid are NaN and then forcing evid and orid to be integers.

In [ ]:
# Drop the nan tuples and then convert evid to int
df = df.dropna(subset=['evid','orid'])
df['evid'] = df['evid'].astype(int)
df['orid'] = df['orid'].astype(int)
print("Size of DataFrame with unassociated arrivals removed=",len(df))

## station metadata
We will need a list of stations eventually to fetch proper metadata.   For now this just validates an algorithm.

In [ ]:
stalist = df['sta'].unique()

In [ ]:
# examine result for validity
print(stalist)

This month doesn't seem to have a net-sta issue but to make this more general one should handle that.   This little block builds a dictionary to cross-reference Antelope naming defined by their netsta table and net:sta used by Earthscope.  Note a complication is the method we called to create the giant dataframe doesn't merge netsta.

There is a strange issue with the nan station name.   That may cause problems downstream.

IMPORTANT:   I hacked in the snetsta table currently in the css30 directory.   The master data repository has arrival data in month directories with the dbmaster data somewhere else.   I'll need to figure that out eventually.  

In [ ]:
df_netdata=anfdb.get_table('snetsta')
# useful to show this
print(df_netdata.columns)

Not obvious and explained only by digging into the antelope schema documents is that fsta is the seed sta name while "sta" for antelope is the composite string used to handle stations with duplicate station names and different net codes.  This little block demonstrates how many of them we have in the snetsta table we loaded.

In [ ]:
print(df_netdata)

In [ ]:
for row in df_netdata.itertuples():
    if row.fsta!=row.sta:
        print(row.fsta,row.sta)

Now that we know the difference between fsta and sta we can build this cross reference. 

In [ ]:
netsta_xref=dict()
for row in df_netdata.itertuples():
    netsta_xref[row.sta] = [row.snet,row.fsta]

In [ ]:
# print a couple examples and verify the length
print("size of cross-reference table=",len(netsta_xref))

By combining the contents of stalist and netsta_xref we can build a bombproof algorithm to fetch station metadata with now standard web servcies.   

## Build subsetted arrivals DataFrame 
We next need to build a subset of the catalog dataframe that includes only teleseismic events.   For this case because the anf data are clean we can use the column of data with the key "delta", which in css3.0 means the distance in degrees.   That is a relatively standard DataFrame operation done in the next frame.

In [ ]:
# there are multiple ways to do the following operation.  This is the clearest to humans
# this query uses stock limits for teleseismic P wave processing for receiver function deconvolution 
# which the tutorial workflow will be doing
# note prefor clause is needed because the method that created df doesn't do that according to the docstring
dftele = df.query('delta>30.0 and delta<100.0 and iphase=="P" and orid==prefor')
print("Size of original catalog = ",len(df))
print("Size of P wave subset = ",len(dftele))

That is a reasonably large data set for a tutorial exercise with a group.   It would seem a monthly will work fine.  I will probably change the month eventually to be sure the data set contains some useful data, but this will certainly do for prototyping.   

I think the next step is pulling out only columns from the DataFrame that are essential to save as metadata.   This will also require using the cross reference even if we don't need it for this month of data.  First, here are the keys printed in a more readable form:

In [ ]:
for k in dftele.keys():
    print(k)

In [ ]:

copylist=['evid','orid','arid','phase','delta','seaz','esaz','timeres','iphase','fm']
rename_list = { 'lat' : 'source_lat',
                'lon' : 'source_lon',
                'depth' : 'source_depth',
                'time' : 'source_time',
                'time_arrival' : 'Ptime',
                'chan' : 'pick_channel',
              }
magkeys=['ms','ml','mb']
# sta and net are handled specially using the cross reference
# note iterrows is not the fastest way to do this but good enough for 
# this application - cleaner syntax for sure for these data
doclist=list()
for index, row in dftele.iterrows():
    doc=dict()
    for k in copylist:
        doc[k] = row[k]
    for k in rename_list.keys():
        newname = rename_list[k]
        val=row[k]
        doc[newname] = val
    for k in magkeys:
        val = row[k]
        # null magnitudes are given a large negative value in antelope 
        # this needs to be more generic if reused in different context
        if val>4.0:   # could be 0 but appropriate for teleseism dataset
            doc[k] = val
    # handle neT:AR code
    css_sta = row.sta
    if css_sta in netsta_xref:
        x = netsta_xref[css_sta]
        net = x[0]
        sta = x[1]
        doc['net'] = net
        doc['sta'] = sta
    else:
        print(f"Warning:  {css_sta} not found in cross reference dictionary created from snetsta")
        print("sta is left unaltered and net is set to TA")
        doc["net"] = "TA"
        doc["sta"] = css_sta
    doclist.append(doc)

In [ ]:
# clean up memory 
del df
del dftele

In [ ]:
# QC results
from mspasspy.util.seismic import print_metadata
print("Size of arrival document list=",len(doclist))
print("Example document with pretty printing")
print_metadata(doclist[0])

## Save to MongoDB
I think the right approach here is to save these data to special mongodb collection.   We can read that later to drive waveform extraction from s3.

In [ ]:
from mspasspy.client import Client
mspass_client=Client()
db = mspass_client.get_database("ANF48")

In [ ]:
out = db.arrival_css30.insert_many(doclist)

In [ ]:
# verify 
n=db.arrival_css30.count_documents({})
print("size of new collection created=",n)

In [ ]:
del doclist

# Get Station Metadata
We know from lots of previous experience that downloading station metadata with obspy is best done as needed.   The data volume is relatively lightweight, the service runs fast, and there is a simple path to load the result to MongoDB found in all previous Earthscope tutorials.   

This version differs a bit from previously only in the way we select what metadata to download.   In this case we need to download station data for all networks that have picks in the arrival_css30 collection we just created.

In [ ]:
netlist=db.arrival_css30.distinct('net')
print(netlist)

There are extras here from when the TA operated stations in Alaska at the same times as the lower 48.   Because I selected only by distance there was nothing to exclude that data.  A good way to keep them out of our data set is to not download the station metadata.  The get_stations run below will do most of that, but I do want to exclude AK (Alaska monitoring network) immediately.   The extra code box before the main one that runs a long time does that.

I'm using a generous time window but limiting the search region to a generous box around the lower 48 stations of the US.  That choice is because the focus here is stations the ANF used to produce their catalog.   

In [ ]:
netlist = sorted(net for net in netlist if isinstance(net, str) and net and net != "AK")
print(netlist)

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
client=Client("Earthscope")

# time range all of 2004 through end of 2015 
ts=UTCDateTime('2004-01-01T00:00:00.0')
starttime=ts
te=UTCDateTime('2016-01-01T00:00:00.0')
latitude_range=[20.0,55.0]
longitude_range=[-130.0,-60]
for net in netlist:
    try:
        inv=client.get_stations(network=net,
                            starttime=ts,
                            endtime=te,
                            minlatitude=latitude_range[0],
                            maxlatitude=latitude_range[1],
                            minlongitude=longitude_range[0],
                            maxlongitude=longitude_range[1],
                            format='xml',
                            channel='BH?',
                            level='response',
                           )
        print("Saving metadata for network=",net)
        db.save_inventory(inv)
    except FDSNNoDataException as ex:
        print("client threw an FDSNNoDataException.  This is the message it posted")
        print(str(ex))
        print("Cannot obtain station metadata for network code=",net)

Finally, get a summary of how many documents were saved in channel and site.

In [ ]:
nsite=db.site.count_documents({})
nchan=db.channel.count_documents({})
print("Number of documents in channel collection=",nchan)
print("Number of documents in site collection=",nsite)

# Save source data
Saving source data is a bit different because I don't wish to save all the source data in the Datascope database.  We could and it wouldn't matter a lot but this is cleaner. There is also a serious complication in handling magnitude data.  That turns out to be problematic with the anf data.   Earlier tests showed I could not use the netmag table, which exists in this database, with mspass join.   What happens is some origin tuples have multiple netmag entries.  That caused duplicates when I had the join command commented out below.   Revision below handles that differently.

In [ ]:
df_src = anfdb.get_table("event")
df_src = anfdb.join(df_src,"origin",join_keys=["evid"])
df_src = df_src[df_src["orid"] == df_src["prefor"]]
# these had to be excluded as they caused duplicates
#df_src = anfdb.join(df_src,"netmag",join_keys=["evid","orid"])
#df_src = df_src[df_src["orid"] == df_src["prefor"]]
print(df_src)

In [ ]:
# similar to above drop rows with prefor null and convert prefor to an int
df_src = df_src.dropna(subset=['prefor'])
df_src['prefor'] = df_src['prefor'].astype(int)
print("Size of source DataFrame with null prefor tuples removed=",len(df_src))

In [ ]:
# use this list to select only orid values of data used 
oridlist=db.arrival_css30.distinct("orid")
print("Number of events with arrivals saved=",len(oridlist))

Now we need to handle the earlier problem I had with netmag.   I'm going to create a scratch netmag collection in MongoDB.   Note I always clear that collection since it is scratch.

In [ ]:
import numpy as np
db.drop_collection("netmag")
df_nm = anfdb.get_table("netmag")
# this is needed to make mongodb cleaner with any NaN values
df_nm = df_nm.replace({np.nan: None})
records = df_nm.to_dict(orient="records")
save_ret = db.netmag.insert_many(records)
print("Number of records saved=",len(save_ret.inserted_ids))
del save_ret

In [ ]:
# This index will speed the queries below
db.netmag.create_index([("evid",1)])

This next block that saves data is not at all generic.   It intentionally discards attributes from the css table that are not needed for this tutorial.

In [ ]:
df_src.keys()

In [ ]:
db.drop_collection("source")
doclist=list()
nmags=0
n=0
for index,row in df_src.iterrows():
    if row.orid in oridlist:
        n += 1
        doc = { "lat" : row.lat,
                "lon" : row.lon, 
                "depth" : row.depth,
                "time" : row.time, 
                "orid" : row.orid,
                "evid" : row.evid,
                "nass" : row.nass,
                "ndef" : row.ndef,
                "auth" : row.auth,
              }
        evid = row.evid
        magdoc = db.netmag.find_one({"evid" : evid})
        if magdoc is not None:
            nmags += 1
            if "magnitude" in magdoc:
                doc["magnitude"] = magdoc["magnitude"]
            if "magtype" in magdoc:
                doc["magtype"] = magdoc["magtype"]

        if row.ml>0.0:
            doc["ml"] = row.ml
        if row.mb>0.0:
            doc["mb"] = row.mb
        if row.ms>0.0:
            doc["ms"] = row.ms
        doclist.append(doc)
insert_result=db.source.insert_many(doclist)                


In [ ]:
print(n,nmags)

In [ ]:
from mspasspy.util.seismic import print_metadata
n=db.source.count_documents({})
print("Number of source documents saved=",n)
print("Typical content")
doc=db.source.find_one()
print_metadata(doc)